# 02 — Validation du pipeline de contrôle qualité

Ce notebook reproduit **exactement** le pipeline de `src/` en version allégée, étape par étape, avec affichage des résultats intermédiaires à chaque phase.

| Étape | Description | Équivalent `src/` |
|-------|-------------|-------------------|
| 1 | Chargement des données de référence | `setup.py` + `loader.py` |
| 2 | Chargement du fichier source | `loader.py` |
| 3 | Filtre sites actifs + filtre pays | `filters.py` |
| 4 | Vérification de localisation (cascade GPS → Adresse) | `rules/localisation.py` |
| 5 | Affichage des résultats | `reporter.py` |

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from difflib import SequenceMatcher

BASE_DIR = Path("..").resolve()
REFERENCE_DIR = BASE_DIR / "data" / "references"
INPUT_DIR = BASE_DIR / "data" / "inputs"

---
## Fonctions utilitaires

_Équivalent de `src/utils.py`_

In [ ]:
def is_empty(value):
    if value is None: return True
    if isinstance(value, float) and pd.isna(value): return True
    if isinstance(value, str) and value.strip() == "": return True
    return False

def is_not_empty(value):
    return not is_empty(value)

def safe_str(value):
    return "" if is_empty(value) else str(value).strip()

def normalize_string(s):
    return "" if is_empty(s) else " ".join(str(s).lower().strip().split())

def extract_department(cp):
    cp = safe_str(cp).zfill(5)
    if cp[:3] in ["971", "972", "973", "974", "976"]: return cp[:3]
    if cp[:2] == "20": return "2A" if cp[2] in "012" else "2B"
    return cp[:2]

def is_in_bbox(lon, lat, bbox):
    return bbox[0] <= lon <= bbox[2] and bbox[1] <= lat <= bbox[3]

def fuzzy_score(s1, s2):
    return SequenceMatcher(None, s1, s2).ratio() * 100

print("✅ Fonctions utilitaires chargées")

---
## Étape 1 — Chargement des données de référence

_Équivalent de `src/setup.py` + `src/loader.py`_

### 1.1 Codes postaux France

In [ ]:
# Charger le CSV et nettoyer les colonnes
df_cp = pd.read_csv(REFERENCE_DIR / "codes_postaux_france.csv", sep=";", dtype=str, encoding="utf-8")

# Nettoyage des colonnes (enlever le #, normaliser les noms)
df_cp.columns = [col.lstrip("#").strip() for col in df_cp.columns]
rename_map = {
    "Code_commune_INSEE": "code_commune_insee",
    "Nom_de_la_commune": "nom_commune",
    "Code_postal": "code_postal",
    "Libellé_d_acheminement": "libelle_acheminement",
    "Ligne_5": "complement_adresse",
}
for old, new in rename_map.items():
    matches = [c for c in df_cp.columns if c.replace("\x92", "").replace("'", "").lower().startswith(old.lower()[:10])]
    if matches:
        df_cp = df_cp.rename(columns={matches[0]: new})

# Construire le dictionnaire CP → [villes]
cp_to_cities = {}
for _, row in df_cp.iterrows():
    cp = safe_str(row.get("code_postal", row.get("Code_postal", ""))).zfill(5)
    city = safe_str(row.get("nom_commune", row.get("Nom_de_la_commune", "")))
    if cp and city:
        cp_to_cities.setdefault(cp, []).append(city) if city not in cp_to_cities.get(cp, []) else None

print(f"✅ {len(cp_to_cities)} codes postaux chargés")
print(f"   Colonnes : {list(df_cp.columns)}")
df_cp.head(3)

### 1.2 Bounding boxes des pays

In [ ]:
with open(REFERENCE_DIR / "bbox_pays.json", "r", encoding="utf-8") as f:
    bbox_pays = json.load(f)

print(f"✅ {len(bbox_pays)} pays chargés")
print(f"   France : {bbox_pays['FRANCE']}")
pd.DataFrame([
    {"pays": k, "lon_min": v["bbox"][0], "lat_min": v["bbox"][1], "lon_max": v["bbox"][2], "lat_max": v["bbox"][3]}
    for k, v in list(bbox_pays.items())[:10]
])

### 1.3 Bounding boxes des départements (générées depuis le GeoJSON)

In [ ]:
# Charger le GeoJSON des départements
with open(REFERENCE_DIR / "departements_france.geojson", "r", encoding="utf-8") as f:
    geojson = json.load(f)

# Extraire toutes les coordonnées d'une géométrie
def extract_coords(geometry):
    coords = []
    if geometry["type"] == "Polygon":
        for ring in geometry["coordinates"]:
            coords.extend([(p[0], p[1]) for p in ring])
    elif geometry["type"] == "MultiPolygon":
        for polygon in geometry["coordinates"]:
            for ring in polygon:
                coords.extend([(p[0], p[1]) for p in ring])
    return coords

# Calculer la bbox de chaque département
bbox_depts = {}
for feat in geojson["features"]:
    code = feat["properties"]["code"]
    nom = feat["properties"]["nom"]
    coords = extract_coords(feat["geometry"])
    if coords:
        lons = [c[0] for c in coords]
        lats = [c[1] for c in coords]
        bbox_depts[code] = {
            "name": nom,
            "bbox": [round(min(lons), 4), round(min(lats), 4), round(max(lons), 4), round(max(lats), 4)]
        }

print(f"✅ {len(bbox_depts)} bounding boxes départements générées")
pd.DataFrame([
    {"code": k, "nom": v["name"], "bbox": str(v["bbox"])}
    for k, v in list(bbox_depts.items())[:5]
])

---
## Étape 2 — Chargement du fichier source

_Équivalent de `src/loader.py → load_source_data()`_

In [ ]:
fichier = INPUT_DIR / "export_Saas_2026-03-03 1.csv"

# Auto-détecter le séparateur
with open(fichier, "r", encoding="utf-8") as f:
    first_line = f.readline()
sep = ";" if first_line.count(";") > first_line.count(",") else ","
print(f"Séparateur détecté : '{sep}'")

df = pd.read_csv(fichier, sep=sep, dtype=str, encoding="utf-8", low_memory=False)
print(f"✅ {len(df)} lignes, {len(df.columns)} colonnes")

# Statistiques des colonnes clés
cols_cles = ["ID_SITE", "PAYS_SITE", "CP_SITE", "VILLE_SITE", "COORD_X_SITE", "COORD_Y_SITE", "STOCK"]
stats = []
for col in cols_cles:
    if col in df.columns:
        nb = df[col].apply(is_not_empty).sum()
        stats.append({"colonne": col, "rempli": nb, "taux": f"{nb/len(df)*100:.1f}%"})
    else:
        stats.append({"colonne": col, "rempli": "ABSENTE", "taux": "-"})
pd.DataFrame(stats)

---
## Étape 3 — Filtres

_Équivalent de `src/filters.py`_

### 3.1 Filtre sites actifs (STOCK=1, pas de résiliation)

In [ ]:
# STOCK = 1
mask_stock = df["STOCK"].astype(str).str.strip() == "1"

# Pas de date de résiliation
mask_resil = df["DT_EFF_RESIL_CNT"].apply(is_empty)

df_actifs = df[mask_stock & mask_resil].copy()

print(f"Lignes initiales     : {len(df)}")
print(f"Sites actifs retenus : {len(df_actifs)}")
print(f"Exclus STOCK ≠ 1     : {(~mask_stock).sum()}")
print(f"Exclus résiliés      : {(~mask_resil).sum()}")

### 3.2 Filtre par pays (France uniquement)

In [ ]:
COUNTRY_FILTER = ["FRANCE"]

if COUNTRY_FILTER:
    mask_pays = df_actifs["PAYS_SITE"].apply(lambda x: safe_str(x).upper() in COUNTRY_FILTER)
    df_filtered = df_actifs[mask_pays].copy()
    print(f"Filtre pays          : {COUNTRY_FILTER}")
    print(f"Avant filtre         : {len(df_actifs)}")
    print(f"Après filtre         : {len(df_filtered)}")
else:
    df_filtered = df_actifs.copy()
    print("Pas de filtre pays")

# Aperçu des colonnes adresse
cols_addr = ["ID_SITE", "PAYS_SITE", "CP_SITE", "VILLE_SITE", "RUE_SITE", "COORD_X_SITE", "COORD_Y_SITE"]
df_filtered[[c for c in cols_addr if c in df_filtered.columns]].head(10)

---
## Étape 4 — Vérification de localisation (cascade GPS → Adresse)

_Équivalent de `src/rules/localisation.py` qui appelle `prerequisites.py`, `gps.py`, `address.py`_

**Logique :**
- GPS existe (non null) → valider GPS → si anomalie, aussi vérifier l'adresse
- GPS absent → valider l'adresse → si aussi KO → site non localisable

In [ ]:
# Liste pour stocker toutes les anomalies
anomalies = []

# Compteurs
count_gps_ok = 0
count_gps_anomaly = 0
count_addr_ok = 0
count_non_loc = 0

for idx, row in df_filtered.iterrows():
    site_id = row.get("ID_SITE")
    contract_id = row.get("NU_CNT")
    site_anomalies = []
    
    # ========================
    # PRÉREQUIS (toujours)
    # ========================
    
    # P-01 : Pays manquant
    if is_empty(row.get("PAYS_SITE")):
        site_anomalies.append({"code": "P-01", "description": "Pays manquant", "gravite": "GRAVE"})
    
    # P-07 : Données techniques manquantes
    cols_tech = ["CAPITAUX_MURS_BATIMENTS", "CAPITAUX_CONTENU", "CAPITAUX_MATERIEL",
                 "CAPITAUX_MARCHANDISES", "CAPITAUX_TT", "SURFACE", "SURFACE_DPDCE"]
    has_tech = any(is_not_empty(row.get(c)) for c in cols_tech if c in row.index)
    if not has_tech:
        site_anomalies.append({"code": "P-07", "description": "Données techniques manquantes", "gravite": "GRAVE"})
    
    # ========================
    # DÉTERMINER LE CHEMIN
    # ========================
    
    lon_raw = row.get("COORD_X_SITE")
    lat_raw = row.get("COORD_Y_SITE")
    has_gps = is_not_empty(lon_raw) or is_not_empty(lat_raw)
    
    gps_errors = 0
    
    if has_gps:
        # ========================
        # CHEMIN GPS
        # ========================
        has_lon = is_not_empty(lon_raw)
        has_lat = is_not_empty(lat_raw)
        
        # G-01 : GPS incomplet
        if has_lon != has_lat:
            site_anomalies.append({"code": "G-01", "description": "GPS incomplet (une seule coordonnée)", "gravite": "LÉGÈRE"})
            gps_errors += 1
        elif has_lon and has_lat:
            # G-02 / G-03 : Format
            lon, lat = None, None
            try:
                lon = float(lon_raw)
                if not (-180 <= lon <= 180):
                    site_anomalies.append({"code": "G-02", "description": f"Longitude {lon} hors limites", "gravite": "GRAVE"})
                    lon = None; gps_errors += 1
            except (ValueError, TypeError):
                site_anomalies.append({"code": "G-02", "description": f"Longitude '{lon_raw}' non numérique", "gravite": "GRAVE"})
                gps_errors += 1
            
            try:
                lat = float(lat_raw)
                if not (-90 <= lat <= 90):
                    site_anomalies.append({"code": "G-03", "description": f"Latitude {lat} hors limites", "gravite": "GRAVE"})
                    lat = None; gps_errors += 1
            except (ValueError, TypeError):
                site_anomalies.append({"code": "G-03", "description": f"Latitude '{lat_raw}' non numérique", "gravite": "GRAVE"})
                gps_errors += 1
            
            # Cohérence si format OK
            if lon is not None and lat is not None:
                country = safe_str(row.get("PAYS_SITE")).upper()
                cp = safe_str(row.get("CP_SITE"))
                
                # G-05 : GPS vs pays
                if country and country in bbox_pays:
                    bp = bbox_pays[country]["bbox"]
                    if not is_in_bbox(lon, lat, bp):
                        # Diagnostics
                        if is_in_bbox(lat, lon, bp):
                            site_anomalies.append({"code": "G-06", "description": "X/Y probablement inversés", "gravite": "INFO"})
                        elif is_in_bbox(-lon, lat, bp) or is_in_bbox(lon, -lat, bp):
                            site_anomalies.append({"code": "G-07", "description": "Signe probablement erroné", "gravite": "INFO"})
                        else:
                            site_anomalies.append({"code": "G-05", "description": f"GPS ({lon},{lat}) hors de {country}", "gravite": "GRAVE"})
                        gps_errors += 1
                
                # G-04 : GPS vs département
                if cp:
                    dept = extract_department(cp)
                    if dept in bbox_depts:
                        if not is_in_bbox(lon, lat, bbox_depts[dept]["bbox"]):
                            site_anomalies.append({"code": "G-04", "description": f"GPS hors département {dept}", "gravite": "GRAVE"})
                            gps_errors += 1
        
        if gps_errors == 0:
            count_gps_ok += 1
        else:
            count_gps_anomaly += 1
    
    # ======================================================
    # CHEMIN ADRESSE - (si GPS absent ou GPS avec anomalies)
    # ======================================================
    
    if not has_gps or gps_errors > 0:
        addr_errors = 0
        cp = row.get("CP_SITE")
        city = row.get("VILLE_SITE")
        country = safe_str(row.get("PAYS_SITE")).upper()
        
        # A-01 : CP manquant
        if is_empty(cp):
            site_anomalies.append({"code": "A-01", "description": "Code postal manquant", "gravite": "GRAVE"})
            addr_errors += 1
        
        # A-02 : Ville manquante
        if is_empty(city):
            site_anomalies.append({"code": "A-02", "description": "Ville manquante", "gravite": "GRAVE"})
            addr_errors += 1
        
        # A-03 : Rue manquante
        has_rue = any(is_not_empty(row.get(c)) for c in ["RUE_SITE", "NUM_RUE_SITE", "ADRESSE_SITE"])
        if not has_rue:
            site_anomalies.append({"code": "A-03", "description": "Rue manquante", "gravite": "LÉGÈRE"})
            addr_errors += 1
        
        # A-04 / A-05 : Cohérence CP / Ville
        if is_not_empty(cp):
            cp_norm = safe_str(cp).zfill(5)
            expected = cp_to_cities.get(cp_norm, [])
            
            if not expected:
                site_anomalies.append({"code": "A-04", "description": f"CP '{cp}' non trouvé dans la référence", "gravite": "GRAVE"})
                addr_errors += 1
            elif is_not_empty(city):
                city_norm = normalize_string(city)
                expected_norm = [normalize_string(c) for c in expected]
                if city_norm not in expected_norm:
                    best = max(expected_norm, key=lambda c: fuzzy_score(city_norm, c))
                    score = fuzzy_score(city_norm, best)
                    i = expected_norm.index(best)
                    if score >= 90:
                        site_anomalies.append({"code": "A-05", "description": f"Probable typo : '{city}' → '{expected[i]}' ({score:.0f}%)", "gravite": "LÉGÈRE"})
                    elif score >= 70:
                        site_anomalies.append({"code": "A-05", "description": f"Ville douteuse : '{city}' vs '{expected[i]}' ({score:.0f}%)", "gravite": "LÉGÈRE"})
                    else:
                        site_anomalies.append({"code": "A-05", "description": f"'{city}' ≠ CP {cp} (attendu: {expected[:3]})", "gravite": "GRAVE"})
                    addr_errors += 1
        
        # Résultat adresse
        if not has_gps:   # Chemin adresse uniquement
            if addr_errors == 0:
                count_addr_ok += 1
            else:
                count_non_loc += 1
                site_anomalies.append({"code": "L-01", "description": "Site non localisable", "gravite": "GRAVE"})
    
    # Ajouter les anomalies du site
    for a in site_anomalies:
        anomalies.append({"ID_SITE": site_id, "NU_CNT": contract_id, **a})

print(f"\n{'='*50}")
print(f"📊 Résultats de localisation :")
print(f"   Sites analysés        : {len(df_filtered)}")
print(f"   ✅ GPS valide          : {count_gps_ok}")
print(f"   ⚠️  GPS avec anomalies : {count_gps_anomaly}")
print(f"   ✅ Adresse valide      : {count_addr_ok}")
print(f"   ❌ Non localisable     : {count_non_loc}")
print(f"   🔍 Anomalies totales   : {len(anomalies)}")

---
## Étape 5 — Résultats

_Équivalent de `src/reporter.py` (sans l'export Excel)_

### 5.1 Détail des anomalies

In [ ]:
df_anomalies = pd.DataFrame(anomalies)

if len(df_anomalies) > 0:
    print(f"📋 {len(df_anomalies)} anomalies détectées")
    display(df_anomalies)
else:
    print("✅ Aucune anomalie détectée")

### 5.2 Résumé par code d'anomalie

In [ ]:
if len(df_anomalies) > 0:
    resume = df_anomalies.groupby(["code", "gravite"]).size().reset_index(name="nombre")
    resume = resume.sort_values("nombre", ascending=False)
    display(resume)
else:
    print("Rien à résumer")

### 5.3 Données enrichies (sites avec flag anomalie)

In [ ]:
# Ajouter une colonne "a_anomalie" au DataFrame source
if len(df_anomalies) > 0:
    sites_avec_anomalie = set(df_anomalies["ID_SITE"].dropna().unique())
else:
    sites_avec_anomalie = set()

df_enriched = df_filtered.copy()
df_enriched["A_ANOMALIE"] = df_enriched["ID_SITE"].apply(lambda x: "OUI" if x in sites_avec_anomalie else "NON")

cols_affichage = ["ID_SITE", "PAYS_SITE", "CP_SITE", "VILLE_SITE", "RUE_SITE", "COORD_X_SITE", "COORD_Y_SITE", "A_ANOMALIE"]
cols_presentes = [c for c in cols_affichage if c in df_enriched.columns]

print(f"Sites avec anomalie(s) : {len(sites_avec_anomalie)}")
print(f"Sites sans anomalie    : {len(df_enriched) - len(sites_avec_anomalie)}")
df_enriched[cols_presentes].head(15)

---
## Conclusion

Ce notebook reproduit le pipeline complet de `src/` :

| Étape | Résultat |
|-------|----------|
| Données de référence | CP, bbox pays, bbox départements chargés ✅ |
| Fichier source | Chargé avec auto-détection séparateur ✅ |
| Filtre sites actifs | STOCK=1 + pas de résiliation ✅ |
| Filtre pays | France uniquement ✅ |
| Localisation cascade | GPS → Adresse avec signalement complet ✅ |
| Résultats | Anomalies affichées en DataFrame ✅ |

→ Tout fonctionne, on peut maintenant exécuter le code final avec `python main.py`.